# FAIR Quantitative Risk Analysis

## Scenario

Unauthorized access to a cloud-hosted patient records database due to compromised third-party vendor credentials.

This notebook implements a FAIR-based quantitative cyber-risk workflow using calibrated uncertainty ranges and vectorized Monte Carlo simulation.

### FAIR Structure

    LEF = TEF × Vulnerability

    Vulnerability = f(Threat Capability, Control Strength)

    LM = Primary Loss + Secondary Loss

    ALE = LEF × LM

The objective is to translate uncertainty into dollar-denominated loss distributions for risk decision-making.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()

sys.path.insert(
    0,
    str(ROOT / "scripts"),
)

from fair_model import (
    build_default_nodes,
    build_risk_model,
)

from run_simulation import (
    run_simulation,
    calculate_metrics,
    loss_exceedance_data,
)

print("Workspace:", ROOT)


## FAIR Factor Calibration

The input ranges are calibrated scenario assumptions rather than organization-specific historical observations.

Ranges are used instead of point estimates so uncertainty remains explicit throughout the analysis.


In [ ]:
factors = pd.read_csv(
    ROOT / "data" / "fair_factors.csv"
)

factors[
    [
        "factor",
        "node_group",
        "min",
        "most_likely",
        "max",
        "unit",
        "distribution",
    ]
]


## Baseline Monte Carlo Simulation

The model runs 50,000 vectorized iterations without looping over individual simulation events.


In [ ]:
ITERATIONS = 50000

baseline_model = build_risk_model(
    build_default_nodes(
        control_strength_multiplier=1.0
    )
)

baseline = run_simulation(
    baseline_model,
    ITERATIONS,
)

baseline_metrics = calculate_metrics(
    baseline
)

baseline_metrics


In [ ]:
pd.Series(
    baseline_metrics
).to_frame(
    "Baseline"
)


## Loss Exceedance Curve

The curve shows the probability that annualized loss exceeds a given financial threshold.


In [ ]:
losses, probabilities = loss_exceedance_data(
    baseline
)

p90 = np.percentile(
    baseline,
    90,
)

p95 = np.percentile(
    baseline,
    95,
)

plt.figure(
    figsize=(10, 6)
)

plt.plot(
    losses,
    probabilities,
    label="Baseline ALE",
)

plt.axvline(
    p90,
    linestyle="--",
    label=f"P90 = ${p90:,.0f}",
)

plt.axvline(
    p95,
    linestyle=":",
    label=f"P95 = ${p95:,.0f}",
)

plt.xlabel(
    "Annualized Loss Exposure ($)"
)

plt.ylabel(
    "Probability of Exceeding Loss"
)

plt.title(
    "FAIR Loss Exceedance Curve"
)

plt.grid(
    alpha=0.25
)

plt.legend()

plt.show()


## Control Sensitivity Analysis

Control Strength is increased by 20% while the remaining assumptions are held constant.

This estimates how materially improved controls change the financial risk distribution.


In [ ]:
improved_model = build_risk_model(
    build_default_nodes(
        control_strength_multiplier=1.20
    )
)

improved = run_simulation(
    improved_model,
    ITERATIONS,
)

improved_metrics = calculate_metrics(
    improved
)

comparison = pd.DataFrame(
    {
        "Baseline": baseline_metrics,
        "Improved Controls": improved_metrics,
    }
)

comparison


In [ ]:
baseline_loss, baseline_prob = loss_exceedance_data(
    baseline
)

improved_loss, improved_prob = loss_exceedance_data(
    improved
)

plt.figure(
    figsize=(10, 6)
)

plt.plot(
    baseline_loss,
    baseline_prob,
    label="Baseline",
)

plt.plot(
    improved_loss,
    improved_prob,
    label="Control Strength +20%",
)

plt.xlabel(
    "Annualized Loss Exposure ($)"
)

plt.ylabel(
    "Probability of Exceeding Loss"
)

plt.title(
    "Control Sensitivity Comparison"
)

plt.grid(
    alpha=0.25
)

plt.legend()

plt.show()


In [ ]:
baseline_p95 = float(
    np.percentile(
        baseline,
        95,
    )
)

improved_p95 = float(
    np.percentile(
        improved,
        95,
    )
)

reduction = (
    (
        baseline_p95
        - improved_p95
    )
    / baseline_p95
    * 100
)

print(
    f"Baseline P95: ${baseline_p95:,.2f}"
)

print(
    f"Improved-control P95: ${improved_p95:,.2f}"
)

print(
    f"P95 reduction: {reduction:.2f}%"
)


## Risk Treatment Recommendation

The preferred treatment is **mitigate**.

Priority improvements include:

- phishing-resistant MFA
- conditional access
- privileged-access management
- time-bounded vendor access
- continuous third-party session monitoring
- stronger credential lifecycle controls
- rapid revocation capability

The treatment decision should compare annualized implementation cost against the modeled reduction in expected and tail loss.


## Assumptions and Limitations

This model uses calibrated scenario estimates rather than organization-specific loss history.

Important limitations include:

- uncertainty in input calibration
- simplified relationships between FAIR factors
- jurisdiction-dependent regulatory effects
- uncertainty in reputational loss
- uncertainty in customer attrition
- no claim that a specific technology produces exactly 20% stronger controls

The model should be recalibrated as better incident, vendor, control-effectiveness, and financial evidence becomes available.
